# Step 1: Import Libraries and Load Dataset

In [1]:
!pip install -q ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Load CDC Diabetes Health Indicators dataset
dataset = fetch_ucirepo(id=891)

df = pd.concat(
    [dataset.data.features, dataset.data.targets],
    axis=1
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (253680, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


## Step 2: Define the Prediction Problem

The objective is to predict whether a participant has high blood pressure.

Target column: HighBP

Task type: Classification

Class 0 represents participants without high blood pressure.
Class 1 represents participants with high blood pressure.

In [2]:
target = "HighBP"

print("Target column:", target)
print("Task type: Classification")

print("\nTarget distribution:")
print(df[target].value_counts())

print("\nTarget percentages:")
print(df[target].value_counts(normalize=True) * 100)

Target column: HighBP
Task type: Classification

Target distribution:
HighBP
0    144851
1    108829
Name: count, dtype: int64

Target percentages:
HighBP
0    57.09989
1    42.90011
Name: proportion, dtype: float64


## Step 3: Select Features and Target

Features are the input columns used to make predictions.

The HighBP column is excluded from the features because it is the target we want to predict.

This prevents target leakage.

In [3]:
# Select input features
X = df.drop(columns=["HighBP", "Diabetes_binary"])

# Select target column
y = df["HighBP"]

print("Features selected:")
print(X.columns.tolist())

print("\nNumber of features:", X.shape[1])
print("Target:", target)

Features selected:
['HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

Number of features: 20
Target: HighBP


## Step 4: Split Training and Testing Data

The dataset is divided into 80% training data and 20% testing data.

The random_state parameter ensures reproducibility.

Stratification maintains the class proportions in both sets.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training features: (202944, 20)
Testing features: (50736, 20)

Training target distribution:
HighBP
0    0.571
1    0.429
Name: proportion, dtype: float64

Testing target distribution:
HighBP
0    0.570995
1    0.429005
Name: proportion, dtype: float64


## Step 5: Train Logistic Regression Model

Logistic Regression is a classification algorithm used to predict categorical outcomes.

A StandardScaler is used to scale the features before training.

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


## Step 6: Evaluate Training and Testing Accuracy

The trained model is evaluated using both training and testing data.

Comparing the scores helps identify whether the model generalizes to unseen data.

In [7]:
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print("Logistic Regression Results")
print("---------------------------")
print("Training Accuracy:", round(train_score, 4))
print("Testing Accuracy:", round(test_score, 4))
print("Accuracy Gap:", round(train_score - test_score, 4))

Logistic Regression Results
---------------------------
Training Accuracy: 0.723
Testing Accuracy: 0.7263
Accuracy Gap: -0.0034


## Step 7: Understand Overfitting

Overfitting occurs when a model learns the training data too closely but performs poorly on unseen data.

A large difference between training and testing accuracy may indicate overfitting.

In [8]:
gap = train_score - test_score

print("Training Accuracy:", round(train_score, 4))
print("Testing Accuracy:", round(test_score, 4))
print("Accuracy Gap:", round(gap, 4))

if gap > 0.10:
    print("The model may be overfitting.")
elif gap >= 0:
    print("The model has a relatively small training-testing gap.")
else:
    print("Testing accuracy is higher than training accuracy.")

Training Accuracy: 0.723
Testing Accuracy: 0.7263
Accuracy Gap: -0.0034
Testing accuracy is higher than training accuracy.


## Step 8: Train a Deliberately Overfitted Decision Tree

A Decision Tree without a maximum depth restriction can grow until it memorizes the training examples.

This experiment demonstrates how an overly complex model can perform differently on training and testing data.

In [9]:
from sklearn.tree import DecisionTreeClassifier

# Create an unrestricted Decision Tree
greedy_model = DecisionTreeClassifier(random_state=42)

# Train the model
greedy_model.fit(X_train, y_train)

# Evaluate both datasets
greedy_train_score = greedy_model.score(X_train, y_train)
greedy_test_score = greedy_model.score(X_test, y_test)

print("Unrestricted Decision Tree Results")
print("-----------------------------------")
print("Training Accuracy:", round(greedy_train_score, 4))
print("Testing Accuracy:", round(greedy_test_score, 4))
print("Accuracy Gap:", round(greedy_train_score - greedy_test_score, 4))

Unrestricted Decision Tree Results
-----------------------------------
Training Accuracy: 0.9687
Testing Accuracy: 0.6421
Accuracy Gap: 0.3266


## Step 9: Calculate the Baseline Accuracy

A baseline model always predicts the most common class in the training dataset.

This provides a simple benchmark to determine whether our trained model learns useful patterns.

In [10]:
from sklearn.dummy import DummyClassifier

# Create the baseline model
baseline = DummyClassifier(strategy="most_frequent")

# Train the baseline model
baseline.fit(X_train, y_train)

# Calculate baseline accuracy
baseline_score = baseline.score(X_test, y_test)

print("Baseline Accuracy:", round(baseline_score, 4))
print("Logistic Regression Accuracy:", round(test_score, 4))

if test_score > baseline_score:
    print("Logistic Regression beats the baseline.")
else:
    print("Logistic Regression does not beat the baseline.")

Baseline Accuracy: 0.571
Logistic Regression Accuracy: 0.7263
Logistic Regression beats the baseline.


## Step 10: Compare All Models

The performance of Logistic Regression, an unrestricted Decision Tree, and the majority-class baseline is compared using training and testing accuracy.

In [11]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Unrestricted Decision Tree",
        "Majority Class Baseline"
    ],
    "Training Accuracy": [
        train_score,
        greedy_train_score,
        baseline.score(X_train, y_train)
    ],
    "Testing Accuracy": [
        test_score,
        greedy_test_score,
        baseline_score
    ]
})

display(results)

,Model,Training Accuracy,Testing Accuracy
0,Logistic Regression,0.722958,0.726348
1,Unrestricted Decision Tree,0.968696,0.642108
2,Majority Class Baseline,0.571000,0.570995


## Step 11: Interpretation of Results

1. Logistic Regression is used as the main classification model.
2. Training accuracy measures performance on the data used for learning.
3. Testing accuracy measures performance on unseen data.
4. A large difference between training and testing accuracy may indicate overfitting.
5. The unrestricted Decision Tree demonstrates the risk of memorizing training data.
6. The majority-class baseline provides a benchmark for evaluating model performance.

The model is considered a baseline because it uses a simple algorithm without extensive hyperparameter tuning.

## Step 12: Conclusion

In this assignment, a classification model was developed using the CDC Diabetes Health Indicators dataset to predict high blood pressure.

Logistic Regression was trained and evaluated using an 80:20 train-test split. An unrestricted Decision Tree was also trained to demonstrate overfitting.

A majority-class baseline was calculated to compare the performance of the trained model.

This assignment demonstrates feature selection, data splitting, model training, evaluation, overfitting, and baseline comparison.